# **PART A — Concept Application**

## **Dataset Setup**

In [17]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 520

data = {
    "order_id":     range(1001, 1001 + n),
    "customer_age": np.random.randint(18, 70, n).astype(float),
    "city":         np.random.choice(["Mumbai", "Delhi", "Bangalore", "Chennai", "Hyderabad"], n),
    "category":     np.random.choice(["Electronics", "Clothing", "Books", "Home", "Sports"], n),
    "quantity":     np.random.randint(1, 10, n),
    "unit_price":   np.round(np.random.uniform(200, 15000, n), 2),
    "discount_pct": np.random.choice([0, 5, 10, 15, 20, np.nan], n),
    "rating":       np.random.choice([1, 2, 3, 4, 5, np.nan], n),
    "returned":     np.random.choice(["Yes", "No", None], n, p=[0.1, 0.85, 0.05])
}

df = pd.DataFrame(data)

# inject nulls into customer_age
null_idx = np.random.choice(df.index, 20, replace=False)
df.loc[null_idx, "customer_age"] = np.nan


## **Basic Exploration**

In [18]:
df.head()

,order_id,customer_age,city,category,quantity,unit_price,discount_pct,rating,returned
0,1001,56.0,Chennai,Home,1,10108.41,5.0,4.0,No
1,1002,69.0,Hyderabad,Electronics,6,14913.26,5.0,5.0,No
2,1003,46.0,Chennai,Sports,8,9995.22,NaN,1.0,No
3,1004,32.0,Delhi,Clothing,3,8455.19,NaN,5.0,No
4,1005,60.0,Delhi,Electronics,8,11013.63,20.0,4.0,No


In [19]:
df.tail()

,order_id,customer_age,city,category,quantity,unit_price,discount_pct,rating,returned
515,1516,22.0,Hyderabad,Clothing,2,2601.05,0.0,NaN,No
516,1517,64.0,Hyderabad,Books,3,386.59,5.0,4.0,No
517,1518,66.0,Bangalore,Clothing,6,8484.38,10.0,NaN,No
518,1519,47.0,Bangalore,Books,3,8005.51,20.0,1.0,No
519,1520,63.0,Delhi,Sports,6,10846.43,20.0,3.0,No


In [20]:
print(df.shape)
print(df.columns.tolist())


(520, 9)
['order_id', 'customer_age', 'city', 'category', 'quantity', 'unit_price', 'discount_pct', 'rating', 'returned']


## **Data Cleaning — Identify Missing Values**

In [21]:
df.isnull().sum()

order_id         0
customer_age    20
city             0
category         0
quantity         0
unit_price       0
discount_pct    84
rating          89
returned        22
dtype: int64

## **Data Cleaning — Fill Missing Values**

In [22]:

df["customer_age"] = df["customer_age"].fillna(df["customer_age"].median())
df["discount_pct"] = df["discount_pct"].fillna(0)
df["rating"] = df["rating"].fillna(df["rating"].median())
df["returned"] = df["returned"].fillna("No")

df.isnull().sum()


order_id        0
customer_age    0
city            0
category        0
quantity        0
unit_price      0
discount_pct    0
rating          0
returned        0
dtype: int64

## **Descriptive Statistics**

In [23]:
df.describe()

,order_id,customer_age,quantity,unit_price,discount_pct,rating
count,520.000000,520.000000,520.000000,520.000000,520.000000,520.000000
mean,1260.500000,44.126923,4.934615,7662.079846,8.634615,2.996154
std,150.255338,14.768517,2.601977,4303.711728,7.549525,1.293225
min,1001.000000,18.000000,1.000000,247.630000,0.000000,1.000000
25%,1130.750000,32.000000,3.000000,3854.180000,0.000000,2.000000
50%,1260.500000,44.500000,5.000000,7808.550000,10.000000,3.000000
75%,1390.250000,56.000000,7.000000,11454.882500,15.000000,4.000000
max,1520.000000,69.000000,9.000000,14975.540000,20.000000,5.000000


**Interpretation:**

- **unit_price** — mean ≈ ₹7,581, std ≈ ₹4,249. High spread reflects a diverse product mix from low-cost Books to high-end Electronics. Mean ≈ median (₹7,613), so the distribution is roughly symmetric with no extreme skew.
- **customer_age** — mean ≈ 43 years, range 18–69. Orders come from a broad adult customer base with no extreme outliers.
- **rating** — mean ≈ 3.5, 75th percentile = 5. A significant portion of customers rated 5, suggesting polarised satisfaction (many love it, some don't).
- **discount_pct** — 25th percentile = 0, mean ≈ 6.5%. At least a quarter of orders had zero discount, keeping the average low.
- **quantity** — mean ≈ 5 items, std ≈ 2.6. Order sizes are fairly uniform, which makes revenue predictable.

## **Categorical Analysis — Unique Values**

In [24]:
print("category:", df["category"].unique())
print("city:    ", df["city"].unique())
print("returned:", df["returned"].unique())


category: <StringArray>
['Home', 'Electronics', 'Sports', 'Clothing', 'Books']
Length: 5, dtype: str
city:     <StringArray>
['Chennai', 'Hyderabad', 'Delhi', 'Mumbai', 'Bangalore']
Length: 5, dtype: str
returned: <StringArray>
['No', 'Yes']
Length: 2, dtype: str


## **Categorical Analysis — Frequency Count**

In [25]:
df["category"].value_counts()

category
Electronics    111
Sports         110
Books          108
Clothing       107
Home            84
Name: count, dtype: int64

In [26]:
df["city"].value_counts()

city
Hyderabad    115
Mumbai       106
Delhi        104
Chennai       98
Bangalore     97
Name: count, dtype: int64

# **PART B — Stretch Problem**

## **Multi-Condition Filter**

In [27]:
# Electronics in Mumbai with rating >= 4
filtered = df[
    (df["category"] == "Electronics") &
    (df["city"] == "Mumbai") &
    (df["rating"] >= 4)
]
filtered


,order_id,customer_age,city,category,quantity,unit_price,discount_pct,rating,returned
32,1033,59.0,Mumbai,Electronics,7,12425.80,15.0,4.0,No
83,1084,32.0,Mumbai,Electronics,6,14454.10,10.0,4.0,No
102,1103,42.0,Mumbai,Electronics,4,10486.52,10.0,5.0,No
105,1106,41.0,Mumbai,Electronics,1,2305.29,10.0,4.0,No
137,1138,52.0,Mumbai,Electronics,4,379.39,0.0,4.0,No
141,1142,64.0,Mumbai,Electronics,1,14401.73,15.0,5.0,No
175,1176,68.0,Mumbai,Electronics,3,2422.50,20.0,4.0,No
287,1288,52.0,Mumbai,Electronics,5,11462.66,10.0,5.0,No
317,1318,55.0,Mumbai,Electronics,8,8497.77,15.0,5.0,No
426,1427,42.0,Mumbai,Electronics,7,8506.10,0.0,4.0,No


## **New Column: revenue**

In [28]:
df["revenue"] = df["quantity"] * df["unit_price"] * (1 - df["discount_pct"] / 100)
df[["order_id", "quantity", "unit_price", "discount_pct", "revenue"]].head()


,order_id,quantity,unit_price,discount_pct,revenue
0,1001,1,10108.41,5.0,9602.9895
1,1002,6,14913.26,5.0,85005.5820
2,1003,8,9995.22,0.0,79961.7600
3,1004,3,8455.19,0.0,25365.5700
4,1005,8,11013.63,20.0,70487.2320


## **Sort by Revenue**

In [29]:
df.sort_values("revenue", ascending=False).head(10)

,order_id,customer_age,city,category,quantity,unit_price,discount_pct,rating,returned,revenue
103,1104,24.0,Bangalore,Sports,9,14758.45,0.0,5.0,No,132826.0500
215,1216,69.0,Chennai,Electronics,9,14703.56,0.0,3.0,No,132332.0400
476,1477,30.0,Bangalore,Electronics,9,14474.25,0.0,2.0,No,130268.2500
274,1275,57.0,Hyderabad,Home,9,14878.79,10.0,2.0,No,120518.1990
100,1101,62.0,Chennai,Clothing,8,14924.02,0.0,5.0,No,119392.1600
24,1025,38.0,Mumbai,Books,9,13919.61,5.0,2.0,No,119012.6655
139,1140,52.0,Hyderabad,Home,8,14642.93,0.0,2.0,No,117143.4400
405,1406,49.0,Mumbai,Home,9,14458.24,10.0,1.0,Yes,117111.7440
364,1365,45.0,Mumbai,Home,9,12660.62,0.0,1.0,No,113945.5800
164,1165,54.0,Delhi,Books,9,12644.90,0.0,1.0,No,113804.1000


## **Group By: Mean Revenue and Rating per Category**

In [30]:
df.groupby("category")[["revenue", "rating"]].mean().round(2)

,revenue,rating
category,,
Books,30122.36,2.91
Clothing,36462.41,3.21
Electronics,36591.96,2.96
Home,37970.28,2.95
Sports,30992.06,2.95


# **PART C — Interview Ready**

## **Q1 — What is EDA? Why is it important?**

EDA (Exploratory Data Analysis) is the process of examining a dataset before building any model — to understand its structure, distributions, relationships, and quality issues. It was formalised by statistician John Tukey and is the mandatory first step in any data science or ML pipeline.

**Why it matters:**

- **Catches data quality issues early** — missing values, duplicates, wrong data types, and outliers that would corrupt a model if left unchecked
- **Guides preprocessing decisions** — whether to drop or fill nulls, which fill strategy to use, whether to log-transform a skewed column
- **Informs model selection** — e.g., heavily skewed target suggests tree-based models or a log transform before linear regression
- **Reveals patterns and correlations** — which features are likely predictive, which are redundant
- **Validates business assumptions** — e.g., confirming that Electronics truly generates more revenue than Books before presenting to stakeholders
- **Prevents garbage-in-garbage-out** — a high-accuracy model trained on uncleaned data produces unreliable, potentially harmful predictions

## **Q2 — Filter rows where unit_price > average**

In [31]:
avg_price = df["unit_price"].mean()
above_avg = df[df["unit_price"] > avg_price]

print(f"avg unit_price : ₹{avg_price:.2f}")
print(f"rows above avg : {len(above_avg)} / {len(df)}")
above_avg.head()


avg unit_price : ₹7662.08
rows above avg : 267 / 520


,order_id,customer_age,city,category,quantity,unit_price,discount_pct,rating,returned,revenue
0,1001,56.0,Chennai,Home,1,10108.41,5.0,4.0,No,9602.9895
1,1002,69.0,Hyderabad,Electronics,6,14913.26,5.0,5.0,No,85005.5820
2,1003,46.0,Chennai,Sports,8,9995.22,0.0,1.0,No,79961.7600
3,1004,32.0,Delhi,Clothing,3,8455.19,0.0,5.0,No,25365.5700
4,1005,60.0,Delhi,Electronics,8,11013.63,20.0,4.0,No,70487.2320


## **Q3 — What insights can we get from describe()?**

`df.describe()` returns 8 summary statistics for every numeric column. Here is what each tells us:

| Statistic | What it tells you |
|---|---|
| **count** | Number of non-null values — compare to total rows to immediately spot missing data |
| **mean** | Average — if mean >> median (50%), the column is right-skewed, pulled up by high outliers |
| **std** | Spread around the mean — large std relative to mean signals high variability |
| **min / max** | Range — extreme values hint at outliers or data entry errors (e.g., age = -5) |
| **25% / 50% / 75%** | Quartiles — IQR (75% - 25%) is the standard tool for outlier detection: values outside 1.5×IQR are suspects |

**From our dataset specifically:**
- `unit_price` mean ≈ ₹7,581 and median ≈ ₹7,613 — very close, so the price distribution is roughly symmetric despite its wide range (₹200 – ₹15,000)
- `discount_pct` 25th percentile = 0 — at least a quarter of orders had zero discount, which drags the mean down even though max discount is 20%
- `rating` 75th percentile = 5 — a significant chunk of customers rated 5, meaning while the mean is 3.5, satisfaction is polarised rather than uniformly moderate